#Loads libraries and data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_PATH = "Dataset.csv"
df = pd.read_csv(DATA_PATH)
print("Raw shape:", df.shape)

Raw shape: (9551, 21)


#Drops rows without cuisines, drops unrated restaurants, resets the index

In [ ]:
df = df.dropna(subset=["Cuisines"]).copy()
df = df[df["Aggregate rating"] > 0].reset_index(drop=True)
df["Cuisines"] = df["Cuisines"].str.strip()
print("After cleaning:", df.shape)

After cleaning: (7394, 21)


#Turns each cuisine list into a TF-IDF vector, splitting on commas

In [ ]:
cuisine_vec = TfidfVectorizer(
    tokenizer=lambda s: [c.strip().lower() for c in s.split(",") if c.strip()],
    lowercase=False,
    token_pattern=None,
)
cuisine_matrix = cuisine_vec.fit_transform(df["Cuisines"])

print("Distinct cuisine tokens:", len(cuisine_vec.vocabulary_))
print("Matrix shape:", cuisine_matrix.shape)

Distinct cuisine tokens: 143
Matrix shape: (7394, 143)


#Scores each restaurant: 0.6 × cuisine similarity + 0.2 × price match + 0.2 × rating, with filters for city, rating and votes

In [ ]:
def recommend(cuisines, price_range, city=None, min_rating=3.0, min_votes=10, top_n=5,
              w_cuisine=0.6, w_price=0.2, w_rating=0.2):
    """Return the top_n restaurants for a user's preferences."""
    user_vec = cuisine_vec.transform([cuisines])
    cuisine_sim = cosine_similarity(user_vec, cuisine_matrix).ravel()

    price_sim = 1 - (df["Price range"] - price_range).abs() / 3
    rating_score = df["Aggregate rating"] / 5

    score = w_cuisine * cuisine_sim + w_price * price_sim + w_rating * rating_score

    mask = (df["Aggregate rating"] >= min_rating) & (df["Votes"] >= min_votes)
    if city:
        mask &= df["City"].str.lower() == city.lower()

    result = df.loc[mask].assign(score=score[mask]).sort_values("score", ascending=False).head(top_n)
    cols = ["Restaurant Name", "City", "Locality", "Cuisines", "Price range",
            "Aggregate rating", "Votes", "score"]
    return result[cols].round(3).reset_index(drop=True)

#Runs sample queries

In [ ]:
recommend("North Indian, Mughlai", price_range=2, city="New Delhi")

,Restaurant Name,City,Locality,Cuisines,Price range,Aggregate rating,Votes,score
0,Dilli BC,New Delhi,Malviya Nagar,"North Indian, Mughlai",2,4.4,84,0.976
1,The Last Mughal (TLM),New Delhi,Shahpur Jat,"North Indian, Mughlai",2,4.4,52,0.976
2,Indian Saffron Co.,New Delhi,Vasant Kunj,"North Indian, Mughlai",2,4.2,62,0.968
3,Shikhara,New Delhi,Greater Kailash (GK) 1,"North Indian, Mughlai",2,4.2,237,0.968
4,Dilli BC,New Delhi,Greater Kailash (GK) 2,"North Indian, Mughlai",2,4.1,267,0.964


In [ ]:
recommend("Italian, Pizza", price_range=3, city="Gurgaon")

,Restaurant Name,City,Locality,Cuisines,Price range,Aggregate rating,Votes,score
0,Fat Lulu's,Gurgaon,Sohna Road,"Pizza, Italian",3,3.7,21,0.948
1,Baking Bad,Gurgaon,DLF Phase 4,"Italian, Pizza",3,3.7,98,0.948
2,California Pizza Kitchen,Gurgaon,"Cyber Hub, DLF Cyber City","Pizza, Italian",4,3.7,980,0.881
3,Crust Bistro,Gurgaon,"Huda City Centre Metro Station, Sector 29, Gur...","Italian, Pizza",2,3.6,144,0.877
4,La Pino'z Pizza,Gurgaon,Sector 31,"Pizza, Italian",2,3.5,22,0.873


In [ ]:
recommend("Cafe, Continental", price_range=2, city="New Delhi")

,Restaurant Name,City,Locality,Cuisines,Price range,Aggregate rating,Votes,score
0,Root'd,New Delhi,Hauz Khas,"Continental, Cafe",3,3.7,18,0.881
1,Cafe Bethak,New Delhi,Delhi University-GTB Nagar,"Cafe, Continental, Chinese",2,3.4,76,0.881
2,Cafe E,New Delhi,"DLF Emporio Mall, Vasant Kunj","Cafe, Continental",3,3.5,83,0.873
3,Pipeline Cafe,New Delhi,Satyaniketan,"Cafe, North Indian, Chinese, Continental",2,4.0,355,0.873
4,Tin Town Caf��,New Delhi,Satyaniketan,"Cafe, North Indian, Continental, Chinese",2,3.9,142,0.869


#Measures cuisine match, price match and average rating for 5 users

In [ ]:
test_users = [
    {"cuisines": "North Indian, Mughlai", "price_range": 2, "city": "New Delhi"},
    {"cuisines": "Italian, Pizza",        "price_range": 3, "city": "Gurgaon"},
    {"cuisines": "Chinese",               "price_range": 1, "city": "Noida"},
    {"cuisines": "Cafe, Desserts",        "price_range": 2, "city": "New Delhi"},
    {"cuisines": "Bakery",                "price_range": 1, "city": "New Delhi"},
]

def evaluate(user, top_n=10):
    recs = recommend(user["cuisines"], user["price_range"], user["city"], top_n=top_n)
    wanted = {c.strip().lower() for c in user["cuisines"].split(",")}
    cuisine_hit = recs["Cuisines"].apply(
        lambda s: len(wanted & {c.strip().lower() for c in s.split(",")}) > 0).mean()
    price_hit = (recs["Price range"] == user["price_range"]).mean()
    return {"user": f'{user["cuisines"]} | price {user["price_range"]} | {user["city"]}',
            "cuisine match": cuisine_hit, "price match": price_hit,
            "avg rating": recs["Aggregate rating"].mean()}

evaluation = pd.DataFrame([evaluate(u) for u in test_users]).set_index("user").round(2)
evaluation

,cuisine match,price match,avg rating
user,,,
"North Indian, Mughlai | price 2 | New Delhi",1.0,1.0,4.06
"Italian, Pizza | price 3 | Gurgaon",1.0,0.4,3.67
Chinese | price 1 | Noida,1.0,0.4,3.54
"Cafe, Desserts | price 2 | New Delhi",1.0,0.5,3.86
Bakery | price 1 | New Delhi,1.0,1.0,3.56


#**Conclusion**

#**Approach:** This content-based recommender represents each restaurant by its cuisines (TF-IDF) and ranks restaurants by a weighted score: 60% cuisine similarity, 20% price match and 20% rating, with filters for city, minimum rating and minimum votes.

#**Results:** In my evaluation with 5 sample users, every recommendation contained at least one requested cuisine (100% cuisine match). Price match varied from 40% to 100%, and average ratings were between 3.5 and 4.1. For the query "North Indian, Mughlai" in New Delhi, all five results matched cuisine and price, with ratings between 4.1 and 4.4.

#**Added test query:** For "Cafe, Continental" in New Delhi, all five results contained both cuisines and three of five matched the requested price range. Two of the top results had ratings of only 3.7 and 3.4, showing that cuisine similarity can outweigh rating in the score.

#**Limitations:** The evaluation is partly circular, because cuisine match is high by design (cuisine carries 60% of the score), and there is no real user data to judge recommendation quality. TF-IDF also lowers the score of restaurants that list many cuisines. The system trades price accuracy for cuisine match.

#**Next steps:** Tune the weights, add distance using latitude and longitude, and use real user histories for collaborative filtering.